# NB07 — Co-expression modules (WGCNA, optional)
**Rule:** WGCNA needs about 100 or more tumours processed the same way. TCGA-GBM primary tumours that are not
IDH-mutant (Toil logCPM) are the **reference** network. The CGGA693 **included** samples (primary IDH-wildtype GBM,
log2 FPKM) are the **test** cohort for module preservation. If the reference has < 100 samples the notebook stops.

Computed: signed network on the top-MAD genes; modules; eigengene correlation with composition traits and the synaptic
score S; one-sided Fisher enrichment of S genes per module (BH); preservation Zsummary in the test cohort
(> 10 strong, 2–10 moderate, < 2 none). Centrality is **not** causal evidence. This notebook is optional: NB10 onward
do not depend on it. Runtime: about 15–45 min.

In [ ]:
import os, sys
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
from gbm_extra import load_expr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
stop_if_smoke("WGCNA needs the full matrices.")
MAT = P("meta", "matrices"); NET = P("network")
N_GENES = int(os.environ.get("GBM_WGCNA_GENES", 5000))          # top-MAD genes in the reference cohort
N_PERMUTATIONS = int(os.environ.get("GBM_WGCNA_PERM", 50))     # raise to 200 for the final run if time allows
MIN_SAMPLES = 100

In [ ]:
R_WGCNA_INSTALL = r'''
options(repos = c(CRAN = "https://cloud.r-project.org"))
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
if (!requireNamespace("WGCNA", quietly = TRUE)) BiocManager::install(c("impute", "preprocessCore", "GO.db", "WGCNA"), ask = FALSE, update = FALSE)
if (!requireNamespace("WGCNA", quietly = TRUE)) stop("WGCNA could not be installed")
cat("WGCNA", as.character(packageVersion("WGCNA")), "\n")
'''
print(run_r(write_r("wgcna_install.R", R_WGCNA_INSTALL), timeout=3600))

In [ ]:
require_files([MAT / "TCGA_logcpm.csv.gz", P("final", "S_candidate_genes.csv")], "Run NB03, NB04 and NB05 first.")
tcga_sheet = pd.read_csv(P("meta", "TCGA_samples.csv"))
use = tcga_sheet.loc[tcga_sheet.included, "sample_id"].tolist()        # primary, not IDH-mutant
ref = load_expr(MAT / "TCGA_logcpm.csv.gz")
use = [s for s in use if s in ref.columns]
ref = ref[use]
print("Reference tumours:", ref.shape[1], "| genes:", ref.shape[0])
if ref.shape[1] < MIN_SAMPLES:
    raise ActionableError(f"Only {ref.shape[1]} reference tumours (< {MIN_SAMPLES}); WGCNA is not justified. Skip NB07 and say so in the methods.")
ref.to_csv(NET / "ref_TCGA_logcpm_tumours.csv.gz")

test_path = NET / "test_CGGA693_included.csv.gz"
cg = MAT / "CGGA693_log2fpkm.csv.gz"
if cg.exists():
    cs = pd.read_csv(P("meta", "CGGA693_samples.csv"))
    t = load_expr(cg)
    keep = [s for s in cs.loc[cs.included, "sample_id"] if s in t.columns]
    t = t[keep]
    t.to_csv(test_path)
    print("Test cohort CGGA693 (included only):", t.shape[1], "samples")
else:
    test_path = NET / "NO_TEST.csv"
    print("CGGA693 matrix not found: preservation will be skipped.")

In [ ]:
S = pd.read_csv(P("final", "S_candidate_genes.csv")).gene.tolist()
z = zscore_df(ref, axis=1)
mk = CFG["markers"]
traits = pd.DataFrame({
    "immune_stromal": score_geneset(z, mk["immune_stromal"]),
    "neuron_identity": score_geneset(z, [g for g in mk["neuron_identity"] if g not in set(S)]),
    "oligodendrocyte": score_geneset(z, mk["oligodendrocyte"]),
    "astrocyte_normal": score_geneset(z, mk["astrocyte_normal"]),
    "S_score": score_geneset(z, S),
})
traits["age"] = tcga_sheet.set_index("sample_id").age.reindex(traits.index).values
traits.index.name = "sample_id"
traits.reset_index().to_csv(NET / "traits_TCGA.csv", index=False)
print(traits.describe().T[["count", "mean", "std"]].round(2).to_string())

In [ ]:
R_WGCNA = r'''
suppressMessages({library(WGCNA); library(data.table)})
options(stringsAsFactors = FALSE); allowWGCNAThreads(nThreads = 2)
args <- commandArgs(TRUE)
ref_f <- args[1]; test_f <- args[2]; traits_f <- args[3]; s_f <- args[4]; out_dir <- args[5]
nperm <- as.integer(args[6]); ngenes <- as.integer(args[7])
rd <- function(f) { x <- fread(f); g <- as.character(x[[1]]); m <- t(as.matrix(x[, -1, with = FALSE])); colnames(m) <- g; m }
ref <- rd(ref_f)
test <- if (file.exists(test_f)) rd(test_f) else NULL
common <- if (!is.null(test)) intersect(colnames(ref), colnames(test)) else colnames(ref)
ref <- ref[, common]
madv <- apply(ref, 2, mad)
keep <- names(sort(madv, decreasing = TRUE))[seq_len(min(ngenes, length(madv)))]
ref <- ref[, keep]
gsg <- goodSamplesGenes(ref, verbose = 0); ref <- ref[gsg$goodSamples, gsg$goodGenes]
sft <- pickSoftThreshold(ref, powerVector = c(1:10, seq(12, 20, 2)), networkType = "signed", verbose = 0)
fi <- sft$fitIndices
ok <- fi$SFT.R.sq >= 0.8 & fi$slope < 0
power <- if (any(ok)) fi$Power[which(ok)[1]] else 12
fwrite(fi, file.path(out_dir, "soft_threshold_fit.csv"))
cat("soft power:", power, "| scale-free criterion met:", any(ok), "\n")
net <- blockwiseModules(ref, power = power, networkType = "signed", TOMType = "signed", minModuleSize = 30,
                        mergeCutHeight = 0.25, numericLabels = TRUE, maxBlockSize = 6000, verbose = 0)
colors <- labels2colors(net$colors)
fwrite(data.frame(gene = colnames(ref), module = colors), file.path(out_dir, "module_assignment.csv"))
cat("modules:", length(setdiff(unique(colors), "grey")), "| unassigned (grey):", sum(colors == "grey"), "\n")
S <- fread(s_f)$gene
universe <- colnames(ref); totalS <- sum(universe %in% S)
rows <- lapply(setdiff(unique(colors), "grey"), function(m) {
  g <- colnames(ref)[colors == m]; a <- sum(g %in% S); b <- length(g) - a
  c <- totalS - a; d <- length(universe) - length(g) - c
  ft <- fisher.test(matrix(c(a, b, c, d), nrow = 2, byrow = TRUE), alternative = "greater")
  data.frame(module = m, size = length(g), n_S = a, expected_S = length(g) * totalS / length(universe), fisher_p = ft$p.value)
})
enr <- do.call(rbind, rows); enr$fisher_FDR <- p.adjust(enr$fisher_p, "BH")
traits <- fread(traits_f); tr <- as.data.frame(traits[match(rownames(ref), traits$sample_id), -1, with = FALSE])
ME <- moduleEigengenes(ref, colors)$eigengenes
cm <- cor(ME, tr, use = "pairwise.complete.obs")
pm <- corPvalueStudent(cm, nSamples = nrow(ref))
long <- data.frame(module = rep(sub("^ME", "", rownames(cm)), ncol(cm)), trait = rep(colnames(cm), each = nrow(cm)),
                   r = as.vector(cm), p = as.vector(pm))
fwrite(long, file.path(out_dir, "eigengene_trait_correlations.csv"))
if (!is.null(test)) {
  tst <- test[, colnames(ref)[colnames(ref) %in% colnames(test)]]
  refp <- ref[, colnames(tst)]
  colp <- colors[match(colnames(tst), colnames(ref))]
  multi <- list(ref = list(data = refp), test = list(data = tst))
  mp <- modulePreservation(multi, multiColor = list(ref = colp), referenceNetworks = 1, nPermutations = nperm,
                           networkType = "signed", randomSeed = 1, verbose = 0)
  zl <- mp$preservation$Z[[1]]; z <- zl[[grep("test", names(zl))[1]]]
  pres <- data.frame(module = rownames(z), Zsummary = z$Zsummary.pres)
  fwrite(pres, file.path(out_dir, "module_preservation_CGGA693.csv"))
  enr <- merge(enr, pres, by = "module", all.x = TRUE)
}
fwrite(enr, file.path(out_dir, "module_S_enrichment.csv"))
cat("WGCNA finished\n")
'''
out = run_r(write_r("wgcna_run.R", R_WGCNA),
            [NET / "ref_TCGA_logcpm_tumours.csv.gz", test_path, NET / "traits_TCGA.csv", P("final", "S_candidate_genes.csv"),
             NET, N_PERMUTATIONS, N_GENES], timeout=10800)
print(out)

In [ ]:
enr = pd.read_csv(NET / "module_S_enrichment.csv").sort_values("fisher_p")
print(enr.head(10).round(4).to_string(index=False))
cor = pd.read_csv(NET / "eigengene_trait_correlations.csv")
piv = cor.pivot(index="module", columns="trait", values="r").reindex(enr.module)
fig, ax = plt.subplots(figsize=(7.5, max(3, 0.32 * len(piv))))
im = ax.imshow(piv.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=40, ha="right")
lab = enr.set_index("module")
ax.set_yticks(range(piv.shape[0]))
ax.set_yticklabels([f"{m} (n={lab.loc[m, 'size']}, S FDR={lab.loc[m, 'fisher_FDR']:.1g})" for m in piv.index], fontsize=7)
fig.colorbar(im, label="Pearson r (eigengene vs trait)"); ax.set_title("Module eigengene correlations (reference: TCGA)")
savefig(fig, "WGCNA_module_trait_heatmap", folder="network"); plt.close(fig)
manifest_update("network", "wgcna", {"reference": "TCGA tumours (not IDH-mutant)", "n_reference_samples": int(ref.shape[1]),
                                     "test": "CGGA693 included", "n_genes": N_GENES, "permutations": N_PERMUTATIONS})
print("\nNB07 complete. Next: NB10 (single cell; defines and freezes M and R).")